In [1]:
!pip install -q ultralytics torch kagglehub numpy

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("fatihkgg/affectnet-yolo-format")

print("Path to dataset files:", path)

Path to dataset files: /home/jovyan/.cache/kagglehub/datasets/fatihkgg/affectnet-yolo-format/versions/2


In [3]:
import numpy

In [4]:
path

'/home/jovyan/.cache/kagglehub/datasets/fatihkgg/affectnet-yolo-format/versions/2'

In [5]:
from ultralytics import YOLO
import torch
import os

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A100 80GB PCIe MIG 3g.40gb


In [6]:
import os
import yaml

# The path to the downloaded dataset root obtained from kagglehub.dataset_download
dataset_root_path = path

# The path to the original data.yaml file
original_data_yaml_path = os.path.join(dataset_root_path, "YOLO_format", "data.yaml")

# Define a new path for the corrected data.yaml file
corrected_data_yaml_path = os.path.join(dataset_root_path, "YOLO_format", "data_corrected.yaml")

# Load the original data.yaml
with open(original_data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

# Update the paths to be relative to the YOLO_format directory where data.yaml resides
# Assuming the structure: YOLO_format/train/images, YOLO_format/valid/images, YOLO_format/test/images
data_config['train'] = os.path.join("train", "images")
data_config['val'] = os.path.join("valid", "images")
# Check if 'test' key exists before modifying it
if 'test' in data_config:
    data_config['test'] = os.path.join("test", "images")

# Save the corrected data.yaml
with open(corrected_data_yaml_path, 'w') as f:
    yaml.safe_dump(data_config, f)

print(f"Corrected data.yaml saved to: {corrected_data_yaml_path}")


Corrected data.yaml saved to: /home/jovyan/.cache/kagglehub/datasets/fatihkgg/affectnet-yolo-format/versions/2/YOLO_format/data_corrected.yaml


In [11]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")


results = model.train(
    data=corrected_data_yaml_path, 
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,
    optimizer="AdamW",
    lr0=1e-2,
    project="EmotionDetection",
    name="YOLOv8n_AffectNet",
    save=True,
    plots=True
)

Ultralytics 8.4.112 🚀 Python-3.12.11 torch-2.13.0+cu130 CUDA:0 (NVIDIA A100 80GB PCIe MIG 3g.40gb, 40192MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jovyan/.cache/kagglehub/datasets/fatihkgg/affectnet-yolo-format/versions/2/YOLO_format/data_corrected.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mo

In [12]:
best_model = YOLO("/home/jovyan/task3 training/runs/detect/EmotionDetection/YOLOv8n_AffectNet/weights/best.pt")

metrics = best_model.val(
    data=corrected_data_yaml_path,
    split="test"
)

Ultralytics 8.4.112 🚀 Python-3.12.11 torch-2.13.0+cu130 CUDA:0 (NVIDIA A100 80GB PCIe MIG 3g.40gb, 40192MiB)
Model summary (fused): 73 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.3±0.0 ms, read: 23.7±16.0 MB/s, size: 12.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /home/jovyan/.cache/kagglehub/datasets/fatihkgg/affectnet-yolo-format/versions/2/YOLO_format/test/labels.cache... 2755 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2755/2755 888.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 173/173 16.5it/s 10.5s0.1s
                   all       2755       2755      0.639      0.737      0.748      0.746
                 Anger        383        383      0.597      0.744      0.737      0.729
              Contempt        332        332      

In [13]:
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

mAP50: 0.7479731570673923
mAP50-95: 0.745590308451745
Precision: 0.6391682824899902
Recall: 0.737370511825968


In [14]:
precision = metrics.box.mp
recall = metrics.box.mr

f1 = 2 * precision * recall / (precision + recall + 1e-16)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

Precision : 0.6392
Recall    : 0.7374
F1 Score  : 0.6848
